# Imports & setup

In [7]:
import yfinance as yf
import polars as pl
from fredapi import Fred
import requests
import os
from dotenv import load_dotenv
from datetime import datetime

load_dotenv()

FRED_API_KEY = os.getenv("FRED_API_KEY")
EIA_API_KEY = os.getenv("EIA_API_KEY")

START_DATE = "2024-01-01"
END_DATE = datetime.today().strftime("%Y-%m-%d")

print(f"Pulling data from {START_DATE} to {END_DATE}")

Pulling data from 2024-01-01 to 2026-07-22


# Market data (yfinance)

In [8]:
tickers = {
    "sp500": "^GSPC",
    "vix": "^VIX",
    "energy_etf": "XLE",
    "oil_etf": "USO",
    "defense_etf": "ITA",
    "airlines_etf": "JETS",
    "staples_etf": "XLP",
    "gold_etf": "GLD",
    "tips_etf": "TIP",
    "dollar_index": "DX-Y.NYB",
    "ten_yr_yield": "^TNX",
}

market_frames = []

for name, ticker in tickers.items():
    df = yf.download(ticker, start=START_DATE, end=END_DATE, progress=False)
    if df.empty:
        print(f"⚠️ No data for {name} ({ticker})")
        continue
    df = df.reset_index()
    df.columns = [c[0] if isinstance(c, tuple) else c for c in df.columns]
    pdf = pl.from_pandas(df[["Date", "Close"]])
    pdf = pdf.rename({"Close": name}).with_columns(pl.col("Date").cast(pl.Date))
    market_frames.append(pdf)

# Join all tickers into one wide dataframe on Date
market_df = market_frames[0]
for f in market_frames[1:]:
    market_df = market_df.join(f, on="Date", how="full", coalesce=True)

market_df.write_csv("../data/raw/market_prices.csv")
print(market_df.shape)
market_df.head()

(642, 12)


Date,sp500,vix,energy_etf,oil_etf,defense_etf,airlines_etf,staples_etf,gold_etf,tips_etf,dollar_index,ten_yr_yield
date,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
2024-01-02,4742.830078,13.2,39.144493,65.769997,123.402298,18.587753,68.106522,190.720001,98.504761,102.199997,3.946
2024-01-03,4704.810059,14.04,39.781971,68.190002,121.720085,17.863686,67.57357,189.130005,98.660782,102.459999,3.907
2024-01-04,4688.680176,14.13,39.084438,67.57,121.710243,18.230679,67.480057,189.320007,98.20195,102.419998,3.991
2024-01-05,4697.240234,13.35,39.116776,69.029999,121.828285,18.617512,67.330475,189.350006,97.981697,102.410004,4.042
2024-01-08,4763.540039,13.08,38.664078,66.400002,120.441208,18.984505,67.826019,187.869995,98.24781,102.209999,4.002


# Macro data (FRED)

In [9]:
fred = Fred(api_key=FRED_API_KEY)

fred_series = {
    "cpi": "CPIAUCSL",
    "core_cpi": "CPILFESL",
    "breakeven_5yr": "T5YIE",
    "breakeven_10yr": "T10YIE",
    "fed_funds_rate": "FEDFUNDS",
    "unemployment": "UNRATE",
    "consumer_sentiment": "UMCSENT",
}

fred_frames = []
for name, series_id in fred_series.items():
    s = fred.get_series(series_id, observation_start=START_DATE, observation_end=END_DATE)
    df = s.reset_index()
    df.columns = ["Date", name]
    pdf = pl.from_pandas(df).with_columns(pl.col("Date").cast(pl.Date))
    fred_frames.append(pdf)

fred_df = fred_frames[0]
for f in fred_frames[1:]:
    fred_df = fred_df.join(f, on="Date", how="full", coalesce=True)

fred_df.write_csv("../data/raw/macro_indicators.csv")
print(fred_df.shape)
fred_df.head()

(676, 8)


Date,cpi,core_cpi,breakeven_5yr,breakeven_10yr,fed_funds_rate,unemployment,consumer_sentiment
date,f64,f64,f64,f64,f64,f64,f64
2024-01-01,309.698,314.319,null,null,5.33,3.7,79.0
2024-01-02,null,null,2.17,2.21,null,null,null
2024-01-03,null,null,2.17,2.2,null,null,null
2024-01-04,null,null,2.18,2.22,null,null,null
2024-01-05,null,null,2.19,2.22,null,null,null


# EIA oil/energy data

In [10]:
def fetch_eia_series(series_path, params_extra=None):
    url = f"https://api.eia.gov/v2/{series_path}/data/"
    params = {
        "api_key": EIA_API_KEY,
        "start": START_DATE,
        "end": END_DATE,
        "sort[0][column]": "period",
        "sort[0][direction]": "asc",
    }
    if params_extra:
        params.update(params_extra)
    r = requests.get(url, params=params)
    r.raise_for_status()
    return r.json()

# Brent crude spot price (daily)
brent = fetch_eia_series(
    "petroleum/pri/spt",
    {"facets[product][]": "EPCBRENT", "frequency": "daily", "data[0]": "value"}
)

brent_df = pl.DataFrame(brent["response"]["data"])
brent_df.write_csv("../data/raw/brent_oil_prices.csv")
print(brent_df.shape)
brent_df.head()

(639, 11)


period,duoarea,area-name,product,product-name,process,process-name,series,series-description,value,units
str,str,str,str,str,str,str,str,str,str,str
"""2024-01-02""","""ZEU""","""NA""","""EPCBRENT""","""UK Brent Crude Oil""","""PF4""","""Spot Price FOB""","""RBRTE""","""Europe Brent Spot Price FOB (D…","""76.24""","""$/BBL"""
"""2024-01-03""","""ZEU""","""NA""","""EPCBRENT""","""UK Brent Crude Oil""","""PF4""","""Spot Price FOB""","""RBRTE""","""Europe Brent Spot Price FOB (D…","""77.18""","""$/BBL"""
"""2024-01-04""","""ZEU""","""NA""","""EPCBRENT""","""UK Brent Crude Oil""","""PF4""","""Spot Price FOB""","""RBRTE""","""Europe Brent Spot Price FOB (D…","""75.79""","""$/BBL"""
"""2024-01-05""","""ZEU""","""NA""","""EPCBRENT""","""UK Brent Crude Oil""","""PF4""","""Spot Price FOB""","""RBRTE""","""Europe Brent Spot Price FOB (D…","""78.31""","""$/BBL"""
"""2024-01-08""","""ZEU""","""NA""","""EPCBRENT""","""UK Brent Crude Oil""","""PF4""","""Spot Price FOB""","""RBRTE""","""Europe Brent Spot Price FOB (D…","""75.47""","""$/BBL"""


# Event timeline (manual, built now)

In [12]:
events = [
    {"date": "2026-02-28", "event": "Operation Epic Fury: US/Israel strikes begin, Khamenei killed", "category": "escalation"},
    {"date": "2026-03-01", "event": "Iran closes Strait of Hormuz to US/Israel-allied shipping", "category": "escalation"},
    {"date": "2026-03-02", "event": "Hezbollah launches missiles into Israel from Lebanon", "category": "escalation"},
    {"date": "2026-03-09", "event": "Mojtaba Khamenei named new Supreme Leader", "category": "political"},
    {"date": "2026-03-17", "event": "Limited Israeli ground invasion of Lebanon begins", "category": "escalation"},
    {"date": "2026-04-08", "event": "First ceasefire agreed (Pakistan-brokered, 2-week deal)", "category": "ceasefire"},
    {"date": "2026-06-10", "event": "Ceasefire collapses; Iran strikes US Gulf bases", "category": "escalation"},
    {"date": "2026-06-11", "event": "Iran re-closes Strait of Hormuz", "category": "escalation"},
    {"date": "2026-06-14", "event": "14-point Islamabad Memorandum signed (digital)", "category": "ceasefire"},
    {"date": "2026-06-18", "event": "US lifts naval blockade on Iran", "category": "de-escalation"},
    {"date": "2026-06-19", "event": "Strait of Hormuz traffic resumes; Israel-Hezbollah ceasefire", "category": "de-escalation"},
]

events_df = pl.DataFrame(events).with_columns(pl.col("date").str.to_date())
events_df.write_csv("../data/raw/event_timeline.csv")
events_df

date,event,category
date,str,str
2026-02-28,"""Operation Epic Fury: US/Israel…","""escalation"""
2026-03-01,"""Iran closes Strait of Hormuz t…","""escalation"""
2026-03-02,"""Hezbollah launches missiles in…","""escalation"""
2026-03-09,"""Mojtaba Khamenei named new Sup…","""political"""
2026-03-17,"""Limited Israeli ground invasio…","""escalation"""
…,…,…
2026-06-10,"""Ceasefire collapses; Iran stri…","""escalation"""
2026-06-11,"""Iran re-closes Strait of Hormu…","""escalation"""
2026-06-14,"""14-point Islamabad Memorandum …","""ceasefire"""
